# AgriSense — Leaf Disease Classification

## 0. Libraries & Reproducibility Setup

In [32]:
import os
import csv
import math
import random
from dataclasses import dataclass
from typing import Callable, Optional

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models

import lightning as pl
from lightning.pytorch.callbacks import BaseFinetuning, ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import CSVLogger
from torchmetrics.classification import MulticlassAccuracy, MulticlassF1Score


def set_seed(seed: int = 42) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    pl.seed_everything(seed, workers=True)


SEED = 42
set_seed(SEED)

Seed set to 42


## 1. Dataset

In [33]:
class LeafImageDataset(Dataset):
    def __init__(self, root_dir: str, transform: Optional[Callable] = None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.classes = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d)) and not d.startswith('.')
        ])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            label = self.class_to_idx[class_name]
            for img_name in sorted(os.listdir(class_dir)):
                if img_name.lower().endswith(valid_extensions):
                    self.image_paths.append(os.path.join(class_dir, img_name))
                    self.labels.append(label)

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)
        return image, label

## 2. DataModule

In [34]:
@dataclass
class DataConfig:
    batch_size: int = 64
    num_workers: int = 4
    pin_memory: bool = True
    shuffle: bool = True
    train_transform: Optional[Callable] = None
    eval_transform: Optional[Callable] = None


class LeafDataModule(pl.LightningDataModule):
    def __init__(self, data_dir: str, config: DataConfig):
        super().__init__()
        self.save_hyperparameters(ignore=['config'])
        self.data_dir = data_dir
        self.config = config

        self.train_dataset: Optional[Dataset] = None
        self.val_dataset: Optional[Dataset] = None
        self.test_dataset: Optional[Dataset] = None

    def setup(self, stage: Optional[str] = None):
        if stage == 'fit' or stage is None:
            self.train_dataset = LeafImageDataset(os.path.join(self.data_dir, 'train'), transform=self.config.train_transform)
            self.val_dataset = LeafImageDataset(os.path.join(self.data_dir, 'validation'), transform=self.config.eval_transform)

        if stage == 'test' or stage is None:
            self.test_dataset = LeafImageDataset(os.path.join(self.data_dir, 'test'), transform=self.config.eval_transform)

    def train_dataloader(self) -> DataLoader:
        return DataLoader(
            self.train_dataset,
            batch_size=self.config.batch_size,
            shuffle=self.config.shuffle,
            num_workers=self.config.num_workers,
            pin_memory=self.config.pin_memory,
            persistent_workers=self.config.num_workers > 0
        )

    def val_dataloader(self) -> DataLoader:
        return DataLoader(
            self.val_dataset,
            batch_size=self.config.batch_size,
            shuffle=False,
            num_workers=self.config.num_workers,
            pin_memory=self.config.pin_memory,
            persistent_workers=self.config.num_workers > 0
        )

    def test_dataloader(self) -> DataLoader:
        return DataLoader(
            self.test_dataset,
            batch_size=self.config.batch_size,
            num_workers=self.config.num_workers,
            shuffle=False,
            pin_memory=self.config.pin_memory,
            persistent_workers=self.config.num_workers > 0
        )

## 3. Callback

In [35]:
class FreezeBackbone(BaseFinetuning):
    def __init__(self, unfreeze_at_epoch: int = 5, fine_tune_lr: float = 1e-4):
        super().__init__()
        self.unfreeze_at_epoch = unfreeze_at_epoch
        self.fine_tune_lr = fine_tune_lr

    def freeze_before_training(self, pl_module: pl.LightningModule):
        self.freeze(pl_module.backbone)

    def finetune_function(self, pl_module: pl.LightningModule, current_epoch: int, optimizer: torch.optim.Optimizer):
        if current_epoch == self.unfreeze_at_epoch:
            self.make_trainable(pl_module.backbone)

            for m in pl_module.backbone.modules():
                if isinstance(m, (nn.BatchNorm2d, nn.SyncBatchNorm)):
                    m.eval()

            for param_group in optimizer.param_groups:
                if param_group.get('is_backbone', False):
                    param_group['lr'] = self.fine_tune_lr

## 4. Model

In [36]:
class LeafClassifier(pl.LightningModule):
    def __init__(self, model_name: str, num_classes: int, learning_rate: float = 1e-3, weight_decay: float = 1e-3, freeze_backbone: bool = True):
        super().__init__()
        self.save_hyperparameters()
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.model_name = model_name

        if model_name == 'mobilenet_v3_small':
            base = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
            self.backbone = nn.Sequential(base.features, base.avgpool)
            in_features = base.classifier[3].in_features
            self.classifier = base.classifier
            self.classifier[3] = nn.Linear(in_features, num_classes)

        elif model_name == 'efficientnet_b0':
            base = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
            self.backbone = nn.Sequential(base.features, base.avgpool)
            in_features = base.classifier[1].in_features
            self.classifier = base.classifier
            self.classifier[1] = nn.Linear(in_features, num_classes)
        else:
            raise ValueError(f"Model {model_name} is not supported!")

        self.criterion = nn.CrossEntropyLoss()
        self.train_acc = MulticlassAccuracy(num_classes=num_classes)
        self.val_acc = MulticlassAccuracy(num_classes=num_classes)
        self.test_acc = MulticlassAccuracy(num_classes=num_classes)
        self.train_f1 = MulticlassF1Score(num_classes=num_classes, average='macro')
        self.val_f1 = MulticlassF1Score(num_classes=num_classes, average='macro')
        self.test_f1 = MulticlassF1Score(num_classes=num_classes, average='macro')

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        self.train_acc(logits, y)
        self.train_f1(logits, y)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train_acc", self.train_acc, on_step=False, on_epoch=True, prog_bar=True)
        self.log("train_f1", self.train_f1, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        self.val_acc(logits, y)
        self.val_f1(logits, y)
        self.log("val_loss", loss, on_epoch=True, prog_bar=True)
        self.log("val_acc", self.val_acc, on_epoch=True, prog_bar=True)
        self.log("val_f1", self.val_f1, on_epoch=True, prog_bar=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        self.test_acc(logits, y)
        self.test_f1(logits, y)
        self.log("test_loss", loss, on_epoch=True, prog_bar=True)
        self.log("test_acc", self.test_acc, on_epoch=True, prog_bar=True)
        self.log("test_f1", self.test_f1, on_epoch=True, prog_bar=True)

    def on_train_epoch_start(self):
        for m in self.backbone.modules():
            if isinstance(m, (nn.BatchNorm2d, nn.SyncBatchNorm)):
                m.eval()

    def configure_optimizers(self):
        backbone_decay, backbone_no_decay = [], []
        head_decay, head_no_decay = [], []

        for name, param in self.named_parameters():
            is_no_decay = any(nd in name for nd in ['bias', 'bn', 'norm'])

            if 'backbone' in name:
                if is_no_decay:
                    backbone_no_decay.append(param)
                else:
                    backbone_decay.append(param)
            else:
                if is_no_decay:
                    head_no_decay.append(param)
                else:
                    head_decay.append(param)

        optimizer_grouped_parameters = [
            {"params": backbone_decay, "weight_decay": self.weight_decay, "lr": self.learning_rate, "is_backbone": True},
            {"params": backbone_no_decay, "weight_decay": 0.0, "lr": self.learning_rate, "is_backbone": True},
            {"params": head_decay, "weight_decay": self.weight_decay, "lr": self.learning_rate, "is_backbone": False},
            {"params": head_no_decay, "weight_decay": 0.0, "lr": self.learning_rate, "is_backbone": False},
        ]

        optimizer_grouped_parameters = [g for g in optimizer_grouped_parameters if len(g["params"]) > 0]
        optimizer = torch.optim.AdamW(optimizer_grouped_parameters)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.trainer.max_epochs,
            eta_min=1e-6
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
                "frequency": 1
            }
        }

## 5. Training & Evaluation

### 5.0 Dataset Selection & Dummy Generator

In [37]:
datasets = {
    "dummy": "../dataset",
}

DATASET_NAME = "dummy"
DATA_DIR = datasets[DATASET_NAME]

if DATASET_NAME == "dummy":
    dummy_classes = ["tomato__early_blight", "tomato__late_blight", "tomato__healthy"]
    split_counts = {"train": 24, "validation": 12, "test": 24}
    for split, count in split_counts.items():
        for cls_name in dummy_classes:
            folder = os.path.join(DATA_DIR, split, cls_name)
            os.makedirs(folder, exist_ok=True)
            for i in range(count):
                img_path = os.path.join(folder, f"img_{i:03d}.jpg")
                if not os.path.exists(img_path):
                    arr = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
                    Image.fromarray(arr).save(img_path)

print(f"Active dataset: '{DATASET_NAME}' -> {os.path.abspath(DATA_DIR)}")

Active dataset: 'dummy' -> /home/nathanaelss/dev/AgriSense/ml/dataset


### 5.1 Data Preparation

In [38]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

config = DataConfig(
    batch_size=64,
    num_workers=4,
    train_transform=train_transform,
    eval_transform=eval_transform,
)

datamodule = LeafDataModule(data_dir=DATA_DIR, config=config)
datamodule.setup()
class_names = datamodule.train_dataset.classes
num_classes = len(class_names)
print(f"Number of classes ({DATASET_NAME}): {num_classes} -> {class_names}")

Number of classes (dummy): 3 -> ['tomato__early_blight', 'tomato__healthy', 'tomato__late_blight']


### 5.2 Model Training

In [ ]:
MODEL_NAME = "mobilenet_v3_small"
UNFREEZE_EPOCH = 5

model = LeafClassifier(
    model_name=MODEL_NAME,
    num_classes=num_classes,
    learning_rate=1e-3,
    weight_decay=1e-3,
)

csv_logger = CSVLogger(save_dir=".", name="lightning_logs")

checkpoint_cb = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename=f"{DATASET_NAME}_{MODEL_NAME}_{{epoch}}-{{val_loss:.4f}}",
)

callbacks = [
    FreezeBackbone(unfreeze_at_epoch=UNFREEZE_EPOCH, fine_tune_lr=1e-4),
    checkpoint_cb,
    EarlyStopping(monitor="val_loss", patience=10, mode="min"),
]

trainer = pl.Trainer(
    max_epochs=30,
    accelerator="auto",
    devices="auto",
    deterministic=True,
    logger=csv_logger,
    callbacks=callbacks,
)

trainer.fit(model, datamodule=datamodule)
log_dir = trainer.logger.log_dir
print(f"Log directory: {log_dir}")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name       | Type               | Params | Mode  | FLOPs
------------------------------------------------------------------
0 | backbone   | Sequential         | 927 K  | train | 0    
1 | classifier | Sequential         | 593 K  | train | 0    
2 | criterion  | CrossEntropyLoss   | 0      | train | 0    
3 | train_acc  | MulticlassAccuracy | 0      | train | 0    
4 | val_acc    | MulticlassAccuracy | 0      | train | 0    
5 | test_acc   | MulticlassAccuracy | 0      | train | 0    
6 | train_f1   | MulticlassF1Score  | 0      | train | 0    
7 | val_f1     | MulticlassF1Score  | 0      | train | 0    
8 | test_f1    | MulticlassF1Score  | 0      | train | 0    
-----------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/nathanaelss/dev/AgriSense/Env/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/nathanaelss/dev/AgriSense/Env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/home/nathanaelss/dev/AgriSense/Env/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:321: The number of training batches (2) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 0:   0%|          | 0/2 [00:00<?, ?it/s]

### 5.3 Training & Validation Curves

In [ ]:
history = {}
with open(os.path.join(log_dir, "metrics.csv"), newline="") as f:
    for row in csv.DictReader(f):
        if row.get("epoch"):
            ep = int(float(row["epoch"]))
            history.setdefault(ep, {}).update({
                k: float(v) for k, v in row.items() if v != "" and k != "epoch"
            })

epochs = sorted(history.keys())
metrics_to_plot = [("Loss", "loss"), ("Accuracy", "acc"), ("Macro F1", "f1")]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (label, key) in zip(axes, metrics_to_plot):
    ax.plot(epochs, [history[e].get(f"train_{key}", np.nan) for e in epochs], marker="o", label="Train")
    ax.plot(epochs, [history[e].get(f"val_{key}", np.nan) for e in epochs], marker="s", label="Validation")
    if max(epochs, default=0) >= UNFREEZE_EPOCH:
        ax.axvline(UNFREEZE_EPOCH, color="gray", linestyle="--", label=f"Unfreeze (Ep {UNFREEZE_EPOCH})")
    ax.set(title=f"{label} ({DATASET_NAME})", xlabel="Epoch", ylabel=label)
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
curves_path = os.path.join(log_dir, f"{DATASET_NAME}_{MODEL_NAME}_training_curves.png")
fig.savefig(curves_path, dpi=150, bbox_inches="tight")
print(f"Saved training curves to: {curves_path}")
plt.show()

### 5.4 Test Evaluation & Classification Report

In [ ]:
trainer.test(model, datamodule=datamodule, ckpt_path="best")

best_ckpt = checkpoint_cb.best_model_path
best_model = LeafClassifier.load_from_checkpoint(best_ckpt) if best_ckpt else model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_model = best_model.to(device).eval()

y_true, y_pred = [], []
sample_images, sample_labels, sample_preds = None, None, None

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(datamodule.test_dataloader()):
        preds = torch.argmax(best_model(images.to(device)), dim=1).cpu()
        y_true.extend(labels.numpy())
        y_pred.extend(preds.numpy())

        if batch_idx == 0:
            sample_images, sample_labels, sample_preds = images.cpu(), labels.cpu(), preds

print(f"\nClassification Report — {DATASET_NAME} ({MODEL_NAME}):")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0))

### 5.5 Test Batch Visualization

In [ ]:
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

num_images = len(sample_images)
cols = int(math.ceil(math.sqrt(num_images)))
rows = int(math.ceil(num_images / cols))

fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.8))
for i, ax in enumerate(np.array(axes).flatten()):
    ax.axis("off")
    if i < num_images:
        img = (sample_images[i] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
        gt, pred = class_names[sample_labels[i]], class_names[sample_preds[i]]
        ax.imshow(img)
        ax.set_title(f"GT: {gt}\nPred: {pred}", color="green" if gt == pred else "red", fontsize=8)

plt.suptitle(f"Test Batch — {DATASET_NAME} ({MODEL_NAME}, {num_images} Images)", fontsize=14, fontweight="bold")
plt.tight_layout()
test_vis_path = os.path.join(log_dir, f"{DATASET_NAME}_{MODEL_NAME}_test_batch_visualization.png")
fig.savefig(test_vis_path, dpi=150, bbox_inches="tight")
print(f"Saved test batch visualization to: {test_vis_path}")
plt.show()